# Finding Simplicial Mori Cone Caps

This tutorial demonstrates how **cyopt** can be used to search for Calabi-Yau triangulations with *simplicial* Mori cone caps — caps whose number of extremal rays equals $h^{1,1}$. A simplicial cap greatly simplifies the analysis of effective curves and divisor classes on the Calabi-Yau manifold.

This application is motivated by [arXiv:2512.00144](https://arxiv.org/abs/2512.00144), which studies the structure of Mori cones and their caps across different triangulations of reflexive polytopes. We use cyopt's genetic algorithm to search for triangulations that minimize the number of extremal rays of the Mori cone cap, comparing GA performance against random sampling across several $h^{1,1}=30$ polytopes from the Kreuzer-Skarke database.

## Setup

In [12]:
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from cytools import fetch_polytopes
import cytools
cytools.config.enable_experimental_features()
from cyopt import GA, RandomSample
from cyopt.frst import frst_optimizer, patch_polytope

print("Imports loaded.")

Imports loaded.


## The Mori Cone Cap at $h^{1,1}=30$

Given a Calabi-Yau threefold $X$ realized as a hypersurface in a toric variety, the **Mori cone** $\overline{\text{NE}}(X)$ is the cone of effective curve classes. CYTools provides `mori_cone_cap(in_basis=True)` which computes a finite set of generators that "cap" the Mori cone.

The Mori cone cap is **simplicial** if it has exactly $h^{1,1}$ extremal rays. This is the minimal possible number. Most triangulations yield non-simplicial caps with many excess generators.

We work at $h^{1,1}=30$, where the DNA spaces are large enough that exhaustive enumeration is infeasible. Each Mori cone cap computation takes ~15-25 seconds, so we compare a short GA run against random sampling to see if the GA can find better triangulations in a limited evaluation budget.

## Target Function

Our target function counts the **extremal rays** of the Mori cone cap. We return the negative count since `frst_optimizer` maximizes the target (fewer extremal rays is better):

$$\text{target}(X) = -n_{\text{extremal rays}}$$

A perfectly simplicial cap achieves $\text{target} = -h^{1,1}$.

In [13]:
def mori_cap_target(cy):
    """Target: number of extremal rays of Mori cone cap (minimised)."""
    cap = cy.mori_cone_cap(in_basis=True)
    n_ext = len(cap.extremal_rays())
    return float(n_ext)

## Loading Polytopes

We select three polytopes at $h^{1,1}=30$ from the Kreuzer-Skarke database, choosing ones with manageable DNA spaces.

In [14]:
candidates = fetch_polytopes(h11=30, limit=20)
print(f"Fetched {len(candidates)} polytopes at h11=30")

# Select polytopes with moderate DNA spaces
selected = []
for i, poly in enumerate(candidates):
    poly.prep_for_optimizers()
    bounds = poly._cyopt_bounds
    total_dna = 1
    for lo, hi in bounds:
        total_dna *= (hi - lo + 1)
    n_int = len(bounds)
    # Pick polytopes with interesting but not huge DNA spaces
    if 1e6 < total_dna < 1e10 and n_int >= 8:
        selected.append((i, poly, bounds, total_dna, n_int))
        print(f"  Poly {i}: {n_int} interesting faces, {total_dna:.2e} DNA")
    if len(selected) >= 3:
        break

print(f"\nSelected {len(selected)} polytopes for comparison")

Fetched 20 polytopes at h11=30
  Poly 0: 8 interesting faces, 1.70e+08 DNA
  Poly 1: 9 interesting faces, 9.75e+06 DNA
  Poly 3: 11 interesting faces, 8.00e+06 DNA

Selected 3 polytopes for comparison


## GA vs Random Sampling

For each polytope, we run:
- **GA**: 5 generations with population size 4 (budget: ~20 unique evaluations)
- **Random Sampling**: 20 random triangulations

Each evaluation takes ~15-25 seconds (dominated by `mori_cone_cap`), so the total runtime is approximately 20 minutes.

In [15]:
results = {}

for idx, poly, bounds, total_dna, n_int in selected:
    h11 = poly.h11('N')
    print(f"\n{'='*60}")
    print(f"Polytope {idx}: h11={h11}, {n_int} interesting faces, {total_dna:.2e} DNA")
    print(f"{'='*60}")
    
    # GA run
    print("\n  GA (P=4, 5 generations)...", flush=True)
    t0 = time.perf_counter()
    ga_opt = frst_optimizer(
        poly,
        mori_cap_target,
        optimizer=GA,
        target_mode='cy',
        seed=42,
        population_size=4,
        mutation_rate=0.3,
        mutation_k=1,
        elitism=1,
        record_history=True,
    )
    ga_result = ga_opt.run(5)
    ga_time = time.perf_counter() - t0
    ga_best = int(ga_result.best_value)
    print(f"    Best extremal rays: {ga_best}  (target: {h11})")
    print(f"    Unique evals: {ga_result.n_evaluations}, time: {ga_time:.0f}s")
    
    # Random sampling run
    print("\n  Random sampling (20 samples)...", flush=True)
    t0 = time.perf_counter()
    rs_opt = frst_optimizer(
        poly,
        mori_cap_target,
        optimizer=RandomSample,
        target_mode='cy',
        seed=42,
    )
    rs_result = rs_opt.run(20)
    rs_time = time.perf_counter() - t0
    rs_best = int(rs_result.best_value)
    print(f"    Best extremal rays: {rs_best}  (target: {h11})")
    print(f"    Unique evals: {rs_result.n_evaluations}, time: {rs_time:.0f}s")
    
    # Collect all evaluated DNA and their extremal ray counts from both caches
    all_ext_rays = []
    for cache in [ga_opt._cache, rs_opt._cache]:
        for dna, val in cache.items():
            all_ext_rays.append(int(val))
    
    results[idx] = {
        'h11': h11,
        'n_int': n_int,
        'total_dna': total_dna,
        'ga_best': ga_best,
        'ga_evals': ga_result.n_evaluations,
        'ga_time': ga_time,
        'ga_history': list(ga_result.history),
        'rs_best': rs_best,
        'rs_evals': rs_result.n_evaluations,
        'rs_time': rs_time,
        'all_ext_rays': all_ext_rays,
    }

print(f"\n{'='*60}")
print("All polytopes done.")

## Results

In [ ]:
# Summary table
print(f"{'Poly':>6} {'h11':>4} {'DNA genes':>10} {'GA best':>8} {'RS best':>8} {'h11 (target)':>13} {'GA evals':>9} {'RS evals':>9}")
print("-" * 78)
for idx, r in results.items():
    print(f"{idx:>6} {r['h11']:>4} {r['n_int']:>10} {r['ga_best']:>8} {r['rs_best']:>8} {r['h11']:>13} {r['ga_evals']:>9} {r['rs_evals']:>9}")

In [ ]:
n_polys = len(results)
fig, axes = plt.subplots(1, n_polys, figsize=(5 * n_polys, 5), sharey=False)
if n_polys == 1:
    axes = [axes]

for ax, (idx, r) in zip(axes, results.items()):
    ext_rays = np.array(r['all_ext_rays'])
    h11 = r['h11']
    
    ax.hist(ext_rays, bins=range(ext_rays.min(), ext_rays.max() + 2),
            alpha=0.7, color='steelblue', edgecolor='white')
    ax.axvline(x=r['ga_best'], color='red', linewidth=2, linestyle='--',
               label=f'GA best ({r["ga_best"]})')
    ax.axvline(x=r['rs_best'], color='orange', linewidth=2, linestyle=':',
               label=f'RS best ({r["rs_best"]})')
    ax.axvline(x=h11, color='green', linewidth=2, linestyle='-.',
               label=f'$h^{{1,1}}$={h11}')
    
    ax.set_xlabel('Extremal Rays', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'Polytope {idx}\n({r["n_int"]} genes, {r["total_dna"]:.0e} DNA)', fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('Mori Cone Cap Extremal Rays at $h^{1,1}=30$', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# GA convergence plots
fig, axes = plt.subplots(1, n_polys, figsize=(5 * n_polys, 4), sharey=False)
if n_polys == 1:
    axes = [axes]

for ax, (idx, r) in zip(axes, results.items()):
    history = r['ga_history']
    ax.plot(range(len(history)), history, 'o-', color='tab:blue', linewidth=2, markersize=5)
    ax.axhline(y=r['h11'], color='green', linestyle=':', alpha=0.7,
               label=f'Simplicial ($h^{{1,1}}$={r["h11"]})')
    ax.axhline(y=r['rs_best'], color='orange', linestyle='--', alpha=0.7,
               label=f'RS best ({r["rs_best"]})')
    ax.set_xlabel('GA Generation', fontsize=11)
    ax.set_ylabel('Best Extremal Rays', fontsize=11)
    ax.set_title(f'Polytope {idx}', fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('GA Convergence: Minimizing Extremal Rays', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Summary

We demonstrated how cyopt's genetic algorithm can search for triangulations with simpler Mori cone cap structure at $h^{1,1}=30$, as motivated by [arXiv:2512.00144](https://arxiv.org/abs/2512.00144):

- The **extremal rays** of the Mori cone cap vary substantially across triangulations of the same polytope. A simplicial cap (with $h^{1,1}$ extremal rays) is the simplest possible.

- Even with a very small evaluation budget (~20 evaluations per method), the GA tends to find triangulations with fewer extremal rays compared to random sampling.

- At $h^{1,1}=30$, each Mori cone cap computation takes ~15-25 seconds, making efficient optimization crucial. The DNA encoding of FRST classes allows the GA to exploit structure in the search space.

- For larger-scale searches, the GA's advantage over random sampling should increase, as the structure-exploiting selection and crossover operators become more valuable in larger DNA spaces.